# Yield Curve PCA and Scenarios

Module: Term Structure and Interest Rate Models

## Lesson summary

Yield curves move as high-dimensional objects, but most daily variation is usually explained by a small number of factors. Principal Component Analysis compresses curve changes into level, slope, and curvature-like shocks that can be used for stress testing and asset-liability management.

## Learning objectives

By the end of this lesson, students should be able to:

- build a constant-maturity yield panel;
- compute yield changes and run PCA;
- interpret the first three principal components as curve shocks;
- construct level, slope, and curvature scenarios;
- explain the limits of PCA-based stress testing.

## PCA representation

Let $\Delta y_t$ be the vector of yield changes across maturities. PCA approximates standardized curve changes with a small number of orthogonal factors:

$$
\Delta y_t \approx b_1 f_{1,t}+b_2 f_{2,t}+b_3 f_{3,t}.
$$

The loading vectors $b_1$, $b_2$, and $b_3$ are interpreted as level, slope, and curvature-like shocks when their shapes support that reading.

## Python setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.term_structure import synthetic_mexican_yield_curve_history, yield_curve_pca

## Deterministic curve history

This synthetic panel is built to behave like a Mexican nominal yield curve history. Replace it with instructor-approved Banxico, Valmer, PIP, or other licensed curve data when live or vendor data are available.

In [ ]:
yield_history = synthetic_mexican_yield_curve_history()
yield_history.tail()

In [ ]:
yield_history.iloc[-60:].plot(figsize=(10, 4), title="Synthetic Constant-Maturity Yield Curves")
plt.xlabel("Date")
plt.ylabel("Yield")
plt.grid(True, alpha=0.3)
plt.show()

## PCA on yield changes

In [ ]:
components, explained = yield_curve_pca(yield_history, n_components=3)

explained[["component", "explained_variance_ratio", "cumulative_variance"]]

In [ ]:
components

## Loading interpretation

The sign of a PCA component is arbitrary. The shape matters:

- PC1 usually behaves like a level move when loadings have similar signs across maturities.
- PC2 usually behaves like a slope move when short and long maturities have opposite signs.
- PC3 usually behaves like a curvature move when the belly differs from the wings.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for component in components.index:
    ax.plot(components.columns, components.loc[component], marker="o", label=component)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("PCA Loadings")
ax.set_xlabel("Tenor")
ax.set_ylabel("Standardized loading")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

## Scenario construction

The scenario below applies stylized basis-point shocks to the latest curve. This is not a regulatory stress test, but it shows how PCA language maps into curve movements.

In [ ]:
latest_curve = yield_history.iloc[-1]
tenor_numbers = np.array([float(col.replace("Y", "")) for col in latest_curve.index])

level_shock = np.full_like(tenor_numbers, 0.010)
slope_shock = np.linspace(0.012, -0.006, len(tenor_numbers))
curvature_shock = -0.008 * np.exp(-((tenor_numbers - 5) / 4) ** 2)

scenarios = pd.DataFrame(
    {
        "base": latest_curve.to_numpy(),
        "level_up": latest_curve.to_numpy() + level_shock,
        "flattening": latest_curve.to_numpy() + slope_shock,
        "belly_rally": latest_curve.to_numpy() + curvature_shock,
    },
    index=latest_curve.index,
)
scenarios

In [ ]:
scenarios.plot(figsize=(10, 4), marker="o", title="Curve Stress Scenarios")
plt.xlabel("Tenor")
plt.ylabel("Yield")
plt.grid(True, alpha=0.3)
plt.show()

## Model limitations

- PCA factors are sample-dependent and can change when the yield-curve regime changes.
- Level, slope, and curvature labels are useful interpretations, not fixed economic laws.
- Scenario shocks should be checked against portfolio exposures and historical plausibility.